# Pandas Übung 3 (Bike Share Daten)

Als Data Scientist musst du nicht immer das Rad neu erfinden. Der große Vorteil von Python ist, dass kluge Leute vor dir viel Energie darauf verwendet haben, das Leben für die nächsten Programmierer einfacher zu machen. Also bitte, mach dir das Leben einfacher und verwende Code, der bereits implementiert wurde, nenne es nicht "Kopieren", sondern "freundliches Ausleihen" des Codes anderer Leute. Wenn du in Zukunft ganze Funktionen oder großartige Grafiken kopierst, vergiss nicht, dem Erfinder Anerkennung zu zollen!

Also auch für diese Übung, wenn du an irgendeiner Stelle nicht weiterkommst, schaue dir gute Lösungen von anderen an und lerne viel von ihnen darüber, wie man diese Probleme noch besser löst.
Hier sind zwei gute Ressourcen für kleine Code-Schnipsel, die sehr hilfreich sein können, wenn man mit DataFrames arbeitet:

- [Sebastian Raschkas "Things in Pandas I Wish I'd Known Earlier"](https://nbviewer.jupyter.org/github/rasbt/python_reference/blob/master/tutorials/things_in_pandas.ipynb)
- [Helpful Python Code Snippets for Data Exploration in Pandas](https://medium.com/@msalmon00/helpful-python-code-snippets-for-data-exploration-in-pandas-b7c5aed5ecb9)
- [Manipulating tabular data with Pandas](https://neuroimaging-data-science.org/content/004-scipy/002-pandas.html)

**Am Ende dieser Sitzung solltest du in der Lage sein**
- Daten mit Pandas zu erkunden, um konzeptionelle Fragen zu beantworten
- Verkettete Befehle für effiziente Einzeiler zu schreiben

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv('data/bike_share_201402_trip_data.csv')

Wie viele Beobachtungen gibt es?

In [3]:
df.shape[0]

144015

Ändere die Spalten, um pythonic zu sein:

- Kleinbuchstaben
- ersetze " " durch `_` als Trennzeichen
- ersetze "#" durch `num`

In [4]:
df.columns = (
    df.columns.str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("#", "num", regex=False)
)
df.columns

Index(['trip_id', 'duration', 'start_date', 'start_station', 'start_terminal',
       'end_date', 'end_station', 'end_terminal', 'bike_num',
       'subscription_type', 'zip_code'],
      dtype='str')

Wie viele Arten von Abonnementoptionen gibt es? Was sind die verschiedenen Abonnementtypen?

In [5]:
pd.Series({
    "num_subscription_types": df["subscription_type"].nunique(),
    "subscription_types": df["subscription_type"].unique().tolist(),
})

num_subscription_types                         2
subscription_types        [Subscriber, Customer]
dtype: object

Was ist die Häufigkeit jeder Abonnementoption?

In [6]:
subscription_counts = df["subscription_type"].value_counts()
subscription_counts

subscription_type
Subscriber    113647
Customer       30368
Name: count, dtype: int64

Bitte stelle die Häufigkeit jeder Abonnementoption mit einem Kreisdiagramm dar:

In [ ]:
subscription_counts.plot(kind="pie", autopct="%1.1f%%", ylabel="", title="Anteil der Abonnementtypen")

Bitte stelle die Häufigkeit jeder Abonnementoption mit einem Balkendiagramm dar:

In [ ]:
subscription_counts.plot(kind="bar", title="Häufigkeit der Abonnementtypen", rot=0)
plt.ylabel("Anzahl Fahrten")

Schaue dir die start_station-Spalte an: Welche 10 Stationen kommen am häufigsten vor?

In [ ]:
df["start_station"].value_counts().head(10)

Schaue dir nun die end_station-Spalte an: Welche 10 Stationen kommen am seltensten vor?

In [ ]:
df["end_station"].value_counts().sort_values().head(10)

Erstelle eine Tabelle, die start_station nach subscription_type segmentiert hat und auch die Zeilen-/Spaltenränder (Zwischensummen) enthält. Wenn du dir nicht sicher bist, wie es geht, schaue dir die Dokumentation für `pd.crosstab()` an.

In [ ]:
pd.crosstab(df["start_station"], df["subscription_type"], margins=True)

Schauen wir uns die Dauer an... Welche Einheit wird hier deiner Meinung nach verwendet?

Wie lang ist die kürzeste Fahrt? Wie viele sind so kurz?

In [ ]:
min_duration = df["duration"].min()
pd.Series({
    "unit_guess": "seconds",
    "shortest_ride": min_duration,
    "num_shortest_rides": (df["duration"] == min_duration).sum(),
})

Was denkst du, passiert mit den kurzen Fahrten?

In [ ]:
print("Die sehr kurzen Fahrten sind wahrscheinlich Testfahrten, Fehlstarts oder sofort beendete Ausleihen.")
print("Dass alle kürzesten Fahrten an derselben Start- und Endstation enden, spricht dafür.")

Was ist die längste Fahrt?

In [ ]:
max_duration = df["duration"].max()
max_duration

In [ ]:
df[df["duration"] == max_duration]

Wie würdest du eine "lange" Fahrt definieren? Wie viele Fahrten sind "lang" gemäß deiner Definition?

In [ ]:
long_trip_threshold = 3600
pd.Series({
    "definition": f">= {long_trip_threshold} seconds (1 hour)",
    "num_long_trips": (df["duration"] >= long_trip_threshold).sum(),
})

Erscheinen die langen Dauern sinnvoll? Warum sind sie so lang? Was könnte uns das über die Nutzer sagen?

In [ ]:
print("Die langen Dauern wirken nur teilweise wie echte Fahrzeiten.")
print("Viele sehr lange Fahrten dürften späte Rückgaben, offene Ausleihen oder administrative Ausreißer sein.")
print("Auffällig ist auch, dass lange Fahrten viel häufiger bei 'Customer' als bei 'Subscriber' vorkommen.")

Plotte die duration-Spalte.

In [ ]:
df["duration"].plot(kind="hist", bins=100, title="Verteilung der Fahrtdauer")
plt.xlabel("Dauer in Sekunden")

Liefert dieser Plot irgendwelche Einblicke?

In [ ]:
print("Nur begrenzt, weil die Verteilung stark rechtsschief ist und wenige extreme Ausreißer den Plot dominieren.")
print("Für echte Einblicke sollte man die Daten auf kürzere Fahrten begrenzen oder nach Nutzergruppen aufteilen.")

Wähle Teilabschnitte der Daten aus, um Plots zu erstellen, die mehr Einblicke bieten.

In [ ]:
short_rides = df[df["duration"] < 3600]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
short_rides["duration"].plot(kind="hist", bins=60, ax=axes[0], title="Fahrten unter 1 Stunde")
axes[0].set_xlabel("Dauer in Sekunden")
short_rides.boxplot(column="duration", by="subscription_type", ax=axes[1])
axes[1].set_title("Dauer nach Abonnementtyp (< 1 Stunde)")
axes[1].set_xlabel("Abonnementtyp")
axes[1].set_ylabel("Dauer in Sekunden")
plt.suptitle("")
plt.tight_layout()

Das Produktteam möchte, dass alle Stationsnamen in Kleinbuchstaben und mit `_` als Trennzeichen sind

`South Van Ness at Market` -> `south_van_ness_at_market`

**VERWENDE KEINE FOR-SCHLEIFE. SIE SIND DAS 👿**

In [ ]:
df["start_station"] = df["start_station"].str.lower().str.replace(" ", "_", regex=False)
df[["start_station"]].head()

In [ ]:
df["end_station"] = df["end_station"].str.lower().str.replace(" ", "_", regex=False)
df[["end_station"]].head()

Nimm jetzt einen Timer und stelle ihn auf 15 Minuten ein. Nimm dir diese Zeit, um die Daten geleitet von deiner eigenen Intuition oder Hypothesen zu erkunden...
> Time Boxing ist ein hilfreicher Ansatz bei der Arbeit mit einem neuen Datensatz, damit du nicht in irgendwelche Kaninchenlöcher fällst.

In [ ]:
duration_summary = df.groupby("subscription_type")["duration"].agg(["count", "mean", "median", "max"])
display(duration_summary)
duration_summary[["mean", "median"]].plot(kind="bar", rot=0, title="Durchschnitts- und Median-Dauer nach Abonnementtyp")
plt.ylabel("Dauer in Sekunden")